# 08 — Final Model Evaluation

## Objective

This notebook reads the dynamic candidate-selection artifact produced by Notebook 07, reconstructs the selected standard-Python pipeline, and evaluates it on later untouched data. It does not assume that Logistic Regression, Random Forest, or XGBoost won.

The operating threshold selected from pooled out-of-fold predictions in Notebook 07 is reused without adjustment. September–October may be shown as a diagnostic validation period, while November–December remains the final holdout estimate.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not bootstrap.is_file():
    bootstrap = Path.cwd() / "import_path.py"
spec = importlib.util.spec_from_file_location("import_path", bootstrap)
ip = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ip)

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

from config import project_config as cfg

RANDOM_SEED = cfg.RANDOM_SEED
TARGET_COLUMN = cfg.TARGET_COLUMN
TRAIN_CAP = 200_000
VALIDATION_CAP = 100_000
FINAL_TRAIN_CAP = 300_000
TEST_CAP = 100_000

print("Dynamic standard-Python evaluation environment loaded.")


## Load the Notebook 07 Contract

The feature manifest defines the raw model inputs and preprocessing columns. The selection artifact identifies the winning algorithm, its tuned hyperparameters, and its cross-validated operating threshold.


In [ ]:
def filesystem_path(path_value):
    path_value = str(path_value)
    return "/dbfs/" + path_value[6:] if path_value.startswith("dbfs:/") else path_value


def read_json(path_value):
    with open(filesystem_path(path_value), "r", encoding="utf-8") as handle:
        return json.load(handle)


feature_manifest = read_json(cfg.MODEL_FEATURE_MANIFEST_PATH)
candidate_selection = read_json(cfg.CANDIDATE_SELECTION_PATH)

SELECTED_MODEL_NAME = candidate_selection["selected_model_name"]
SELECTED_MODEL_PARAMETERS = candidate_selection["selected_model_parameters"]
SELECTED_DECISION_THRESHOLD = float(
    candidate_selection.get(
        "selected_decision_threshold",
        feature_manifest.get("selected_decision_threshold", 0.5),
    )
)

MODEL_INPUT_COLUMNS = list(feature_manifest["model_input_columns"])
CATEGORICAL_COLUMNS = list(feature_manifest["categorical_columns"])
NUMERICAL_COLUMNS = list(feature_manifest["numerical_columns"])

print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected threshold: {SELECTED_DECISION_THRESHOLD:.4f}")
print(f"Raw input features: {len(MODEL_INPUT_COLUMNS)}")


## Prepare Bounded, Chronological Samples

The samples preserve the natural class distribution. Sampling is deterministic and occurs only after the chronological periods have been defined. The test period is never used for tuning, threshold selection, or model choice.


In [ ]:
def bounded_pandas(table_name, maximum_rows, seed):
    dataframe = spark.table(table_name).select(
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    )
    row_count = dataframe.count()
    if row_count > maximum_rows:
        dataframe = dataframe.orderBy(F.rand(seed)).limit(maximum_rows)
    result = dataframe.toPandas()
    print(f"{table_name}: {len(result):,} of {row_count:,} rows loaded")
    return result


train_pdf = bounded_pandas(cfg.MODELING_TRAIN_HIST_TABLE, TRAIN_CAP, RANDOM_SEED)
validation_pdf = bounded_pandas(
    cfg.MODELING_VALIDATION_HIST_TABLE,
    VALIDATION_CAP,
    RANDOM_SEED + 1,
)
test_pdf = bounded_pandas(cfg.MODELING_TEST_HIST_TABLE, TEST_CAP, RANDOM_SEED + 2)


## Reconstruct the Selected Pipeline

All three algorithms use the same imputation and one-hot encoding. The class-imbalance strategy saved by Notebook 07 is applied only to the fitting data. Validation and holdout observations retain their natural distribution.


In [ ]:
def make_preprocessor():
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=20,
            ),
        ),
    ])
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    return ColumnTransformer([
        ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
        ("numerical", numerical_pipeline, NUMERICAL_COLUMNS),
    ])


def split_strategy(parameters):
    parameters = dict(parameters)
    strategy = parameters.pop("imbalance_strategy", "natural")
    return parameters, strategy


def undersample(X, y, seed):
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y)
    positive = np.flatnonzero(y_array == 1)
    negative = np.flatnonzero(y_array == 0)
    if len(positive) == 0 or len(negative) == 0:
        return X.reset_index(drop=True), pd.Series(y_array)
    keep_negative = rng.choice(negative, size=min(len(negative), len(positive)), replace=False)
    keep = np.concatenate([positive, keep_negative])
    rng.shuffle(keep)
    return X.iloc[keep].reset_index(drop=True), pd.Series(y_array[keep])


def build_classifier(model_name, parameters, y_train):
    parameters = dict(parameters)
    if model_name == "Logistic Regression":
        max_iter = max(5_000, int(parameters.pop("max_iter", 5_000)))
        return LogisticRegression(
            **parameters,
            max_iter=max_iter,
            random_state=RANDOM_SEED,
        )
    if model_name == "Random Forest":
        return RandomForestClassifier(
            **parameters,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    if model_name == "XGBoost":
        return XGBClassifier(
            **parameters,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    raise ValueError(f"Unsupported selected model: {model_name}")


def fit_selected_pipeline(training_pdf):
    parameters, strategy = split_strategy(SELECTED_MODEL_PARAMETERS)
    X_train = training_pdf[MODEL_INPUT_COLUMNS].copy()
    y_train = training_pdf[TARGET_COLUMN].astype(int).copy()

    if strategy == "undersample":
        X_train, y_train = undersample(X_train, y_train, RANDOM_SEED)
    elif strategy == "class_weight":
        if SELECTED_MODEL_NAME in {"Logistic Regression", "Random Forest"}:
            parameters["class_weight"] = "balanced"
        elif SELECTED_MODEL_NAME == "XGBoost":
            negatives = max(1, int((y_train == 0).sum()))
            positives = max(1, int((y_train == 1).sum()))
            parameters["scale_pos_weight"] = negatives / positives

    classifier = build_classifier(SELECTED_MODEL_NAME, parameters, y_train)
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("classifier", classifier),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline, strategy, len(y_train)


## Evaluate Probabilities and the Stored Operating Threshold

Delayed-flight Recall is the fraction of truly delayed flights detected: `TP / (TP + FN)`. The ordinary weighted Recall shown in some tables is a class-frequency-weighted summary and is not the same quantity. Brier Score evaluates probability calibration; lower values are better. Top-10% Recall and Lift measure operational concentration among the highest-risk flights.


In [ ]:
def evaluate_binary(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, predictions, average="weighted", zero_division=0
    )
    delay_precision, delay_recall, delay_f1, _ = precision_recall_fscore_support(
        y_true, predictions, average="binary", pos_label=1, zero_division=0
    )
    top_count = max(1, int(np.ceil(0.10 * len(y_true))))
    top_indices = np.argsort(-probabilities)[:top_count]
    total_delays = max(1, int(y_true.sum()))
    top_recall = float(y_true[top_indices].sum() / total_delays)
    prevalence = max(np.mean(y_true), 1e-12)
    top_lift = float(np.mean(y_true[top_indices]) / prevalence)
    return {
        "ACCURACY": float(accuracy_score(y_true, predictions)),
        "PRECISION": float(weighted_precision),
        "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "ROC_AUC": float(roc_auc_score(y_true, probabilities)),
        "PR_AUC": float(average_precision_score(y_true, probabilities)),
        "BRIER_SCORE": float(brier_score_loss(y_true, probabilities)),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
        "TOP_10_RECALL": top_recall,
        "TOP_10_LIFT": top_lift,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "THRESHOLD": float(threshold),
    }, predictions


validation_pipeline, imbalance_strategy, validation_fit_rows = fit_selected_pipeline(train_pdf)
validation_probabilities = validation_pipeline.predict_proba(
    validation_pdf[MODEL_INPUT_COLUMNS]
)[:, 1]
validation_metrics, validation_predictions = evaluate_binary(
    validation_pdf[TARGET_COLUMN],
    validation_probabilities,
    SELECTED_DECISION_THRESHOLD,
)

display(spark.createDataFrame([{
    "MODEL": SELECTED_MODEL_NAME,
    "DATASET": "September–October validation",
    **validation_metrics,
}]))


## Final Holdout Evaluation

After the diagnostic validation check, the selected pipeline is refitted on the available pre-holdout development data. The untouched November–December sample provides the definitive performance estimate carried into the later notebooks.


In [ ]:
combined_pdf = pd.concat([train_pdf, validation_pdf], ignore_index=True)
if len(combined_pdf) > FINAL_TRAIN_CAP:
    combined_pdf = combined_pdf.sample(
        FINAL_TRAIN_CAP,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

selected_model_pipeline, imbalance_strategy, final_fit_rows = fit_selected_pipeline(combined_pdf)
test_probabilities = selected_model_pipeline.predict_proba(test_pdf[MODEL_INPUT_COLUMNS])[:, 1]
test_metrics, test_predictions = evaluate_binary(
    test_pdf[TARGET_COLUMN],
    test_probabilities,
    SELECTED_DECISION_THRESHOLD,
)

display(spark.createDataFrame([{
    "MODEL": SELECTED_MODEL_NAME,
    "DATASET": "November–December holdout",
    **test_metrics,
}]))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

matrix = np.array([[test_metrics["TN"], test_metrics["FP"]], [test_metrics["FN"], test_metrics["TP"]]])
axes[0].imshow(matrix, cmap="Blues")
axes[0].set_title("Holdout Confusion Matrix")
axes[0].set_xticks([0, 1], ["Predicted on time", "Predicted delayed"])
axes[0].set_yticks([0, 1], ["Actual on time", "Actual delayed"])
for row in range(2):
    for column in range(2):
        axes[0].text(column, row, f"{matrix[row, column]:,}", ha="center", va="center")

fpr, tpr, _ = roc_curve(test_pdf[TARGET_COLUMN], test_probabilities)
axes[1].plot(fpr, tpr, label=f"AUC={test_metrics['ROC_AUC']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Holdout ROC Curve")
axes[1].legend()

precision_values, recall_values, _ = precision_recall_curve(test_pdf[TARGET_COLUMN], test_probabilities)
axes[2].plot(recall_values, precision_values, label=f"AP={test_metrics['PR_AUC']:.3f}")
axes[2].set_title("Holdout Precision–Recall Curve")
axes[2].legend()
plt.tight_layout()
plt.show()


## Persist the Fitted Python Model and Evaluation Metadata

Notebook 09 and Notebook 10 load this exact fitted pipeline. This eliminates the former Logistic Regression surrogate and Spark-ML-only loading path.


In [ ]:
model_directory = filesystem_path(cfg.SELECTED_MODEL_PATH)
os.makedirs(model_directory, exist_ok=True)
model_bundle_path = os.path.join(model_directory, "model_bundle.joblib")

model_bundle = {
    "pipeline": selected_model_pipeline,
    "model_name": SELECTED_MODEL_NAME,
    "model_parameters": SELECTED_MODEL_PARAMETERS,
    "decision_threshold": SELECTED_DECISION_THRESHOLD,
    "imbalance_strategy": imbalance_strategy,
    "feature_manifest": feature_manifest,
    "candidate_selection": candidate_selection,
}
joblib.dump(model_bundle, model_bundle_path)

selected_model_metadata = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selected_decision_threshold": SELECTED_DECISION_THRESHOLD,
    "modeling_implementation": "standard_python",
    "model_bundle_path": model_bundle_path,
    "imbalance_strategy": imbalance_strategy,
    "final_fit_rows": final_fit_rows,
    "holdout_period": "November–December 2025",
}

selected_model_metrics = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selection_policy": candidate_selection.get("selection_rule"),
    "default_decision_threshold": 0.5,
    "selected_validation_threshold": SELECTED_DECISION_THRESHOLD,
    "metric_source": "untouched November–December holdout",
    **{key.lower(): value for key, value in test_metrics.items()},
    "validation_period_metrics": {
        key.lower(): value for key, value in validation_metrics.items()
    },
    "tuning_validation_metrics": candidate_selection.get("selected_cv_metrics", {}),
}

for path_value, payload in [
    (cfg.SELECTED_MODEL_METADATA_PATH, selected_model_metadata),
    (cfg.SELECTED_MODEL_METRICS_PATH, selected_model_metrics),
]:
    path = filesystem_path(path_value)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=4)

print(f"Saved fitted model bundle: {model_bundle_path}")
print(f"Saved evaluation metrics: {cfg.SELECTED_MODEL_METRICS_PATH}")


## Evaluation Handoff

The downstream notebooks now receive the dynamically selected fitted Python pipeline, its stored decision threshold, calibrated probability metrics, and unchanged raw historical feature contract. Notebook 09 explains the actual selected model; Notebook 10 uses the same bundle for operational scoring.
